# Model CPU 模型统一准备
选择 CPU 运行时；本 Notebook 不执行推理，也不需要 GPU。先运行配置单元查看文件大小，再运行下载单元。请确保 Colab 挂载的是网页使用的同一个 Google Drive 账户。
默认准备小型聊天、Embedding、Reranker、GOT-OCR、Chronos 和 TimesFM。需要时关闭对应开关；不会下载所有量化版本，也不会下载大型视频权重。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install huggingface_hub==0.34.4
from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path
import hashlib, json, time
PREPARE_CHAT = True
PREPARE_EMBEDDING = True
PREPARE_RERANKER = True
PREPARE_OCR = True
PREPARE_CHRONOS = True
PREPARE_TIMESFM = True
vault = Path('/content/drive/MyDrive/AI-Model-Vault')
specs = [
    (PREPARE_CHAT, 'Qwen/Qwen3-0.6B-GGUF', 'llm/Qwen3-0.6B-GGUF'),
    (PREPARE_EMBEDDING, 'Qwen/Qwen3-Embedding-0.6B-GGUF', 'rag/Qwen__Qwen3-Embedding-0.6B-GGUF'),
    (PREPARE_RERANKER, 'ggml-org/Qwen3-Reranker-0.6B-Q8_0-GGUF', 'rag/ggml-org__Qwen3-Reranker-0.6B-Q8_0-GGUF'),
    (PREPARE_OCR, 'stepfun-ai/GOT-OCR-2.0-hf', 'ocr/stepfun-ai__GOT-OCR-2.0-hf'),
    (PREPARE_CHRONOS, 'amazon/chronos-2', 'timeseries/amazon__chronos-2'),
    (PREPARE_TIMESFM, 'google/timesfm-2.0-500m-pytorch', 'timeseries/google__timesfm-2.0-500m-pytorch'),
]
exact_files = {
    'Qwen/Qwen3-0.6B-GGUF': 'Qwen3-0.6B-Q8_0.gguf',
    'Qwen/Qwen3-Embedding-0.6B-GGUF': 'Qwen3-Embedding-0.6B-Q8_0.gguf',
    'ggml-org/Qwen3-Reranker-0.6B-Q8_0-GGUF': 'qwen3-reranker-0.6b-q8_0.gguf',
}
api = HfApi()
plans = []
for enabled, repo, relative in specs:
    if not enabled: continue
    info = api.model_info(repo, files_metadata=True)
    selected = [f for f in info.siblings if f.rfilename.endswith(('.json', '.safetensors', '.model', '.txt', '.tiktoken')) or f.rfilename == 'torch_model.ckpt']
    if repo in exact_files: selected = [f for f in info.siblings if f.rfilename == exact_files[repo]]
    if not selected: raise RuntimeError('No supported model files: '+repo)
    print(repo, 'revision', info.sha, 'bytes', sum(f.size or 0 for f in selected))
    plans.append((repo, relative, info.sha, selected))
print('Review the estimated sizes above before running the download cell.')


In [ ]:
for repo, relative, revision, files in plans:
    destination = vault / relative
    destination.mkdir(parents=True, exist_ok=True)
    manifest = []
    for item in files:
        local = Path(hf_hub_download(repo, item.rfilename, revision=revision, local_dir=destination))
        if item.size is not None and local.stat().st_size != item.size:
            raise RuntimeError('Size mismatch: '+item.rfilename)
        digest = hashlib.sha256()
        with local.open('rb') as stream:
            while block := stream.read(8*1024*1024): digest.update(block)
        actual = digest.hexdigest()
        lfs = item.lfs
        expected = getattr(lfs, 'sha256', None)
        if expected is None and isinstance(lfs, dict): expected = lfs.get('sha256')
        if expected and actual != expected: raise RuntimeError('SHA256 mismatch: '+item.rfilename)
        manifest.append({'path':item.rfilename,'size':local.stat().st_size,'sha256':actual,'lfs_verified':bool(expected)})
    ready = {'schema_version':1,'repo':repo,'revision':revision,'status':'weights_verified_not_inference_tested','verified_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'files':manifest}
    (destination/'MODEL_READY.json').write_text(json.dumps(ready,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Verified weights:', destination)
print('Rescan the Model website, check the model package, then run a real task.')
